
# 4. Final recommendation system

End-to-end recommender that combines the cleaned features (notebook 1), LSH search (notebook 2), and clustering prior (notebook 3). We expose both a high-quality exact-cosine mode and a faster LSH mode, with per-track explanations.

**Contents**
- [4.1 Data & artifacts](#41-data--artifacts)
- [4.2 Helpers for lookups and normalization](#42-helpers-for-lookups-and-normalization)
- [4.3 LSH index (from notebook 2)](#43-lsh-index-from-notebook-2)
- [4.4 Cluster labels for reranking (from notebook 3)](#44-cluster-labels-for-reranking-from-notebook-3)
- [4.5 Recommendation pipelines: exact vs. LSH](#45-recommendation-pipelines-exact-vs-lsh)
- [4.6 Demo: exact cosine vs. LSH](#46-demo-exact-cosine-vs-lsh)
- [4.7 Conclusion](#47-conclusion)



## 4.1 Data & artifacts

Load the cleaned metadata and weighted feature matrix from `main/artifacts`, plus the precomputed LSH signatures. Paths mirror the earlier notebooks so everything runs from the same artifact set.


In [151]:
from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
import json

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)

RANDOM_STATE = 42

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR
while not (PROJECT_ROOT / "artifacts").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
paths = {
    "full_df": ARTIFACT_DIR / "final_cleaned_dataset.csv",
    "features_npz": ARTIFACT_DIR / "track_features.npz",
    "lsh_signatures": ARTIFACT_DIR / "lsh_signatures.npz",
}

for label, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(path)

full_df = pd.read_csv(paths["full_df"])
features_npz = np.load(paths["features_npz"], allow_pickle=True)

track_ids = features_npz["track_ids"].astype(str)
all_features = features_npz["features"].astype(np.float32)
all_weighted = features_npz["features_weighted"].astype(np.float32)

full_df["track_id"] = full_df["track_id"].astype(str)

trackid_to_idx = {tid: i for i, tid in enumerate(track_ids)}
full_df = full_df[full_df["track_id"].isin(trackid_to_idx)].reset_index(drop=True)
row_order = full_df["track_id"].map(trackid_to_idx).to_numpy(dtype=int)

all_features = all_features[row_order]
all_weighted = all_weighted[row_order]
track_ids = full_df["track_id"].to_numpy()

feature_cols = [f"feature_{i:03d}" for i in range(all_weighted.shape[1])]
features_df = pd.DataFrame(all_weighted, columns=feature_cols)
features_df.insert(0, "track_id", track_ids)

id_to_index = {tid: i for i, tid in enumerate(track_ids)}
index_to_id = {i: tid for tid, i in id_to_index.items()}

# Pre-normalize feature matrix for cosine similarity
norms = np.linalg.norm(all_weighted, axis=1, keepdims=True)
all_weighted_unit = all_weighted / np.clip(norms, 1e-12, None)

full_df.head()


,track_id,artists,track_name,track_genre,search_string,duration_ms,tempo,tempo_log,duration_minutes,tags_clean_text,acousticness,danceability,energy,instrumentalness,liveness,loudness,speechiness,valence,lfm_playcount,lfm_listeners,log_playcount,log_listeners,log_plays_per_listener,plays_per_listener,tag_count
0,3nqQXoyQOWXiESFLlDF1hG,Sam Smith;Kim Petras,Unholy (feat. Kim Petras),dance,"Unholy (feat. Kim Petras) - Sam Smith, Kim Petras",156943,131.121,4.876121,2.615717,pop hyperpop electropop electronic pop rap,0.01300,0.714,0.472,0.000005,0.2660,-7.375,0.0864,0.238,10995250.0,920570.0,16.212974,13.732749,2.560629,11.943959,5
1,4uUG5RXrOk84mYEfFvj3cK,David Guetta;Bebe Rexha,I'm Good (Blue),dance,"I'm Good (Blue) - David Guetta, Bebe Rexha",175238,128.040,4.852343,2.920633,house electronic dance electro house 2022,0.00383,0.561,0.965,0.000007,0.3710,-3.673,0.0343,0.304,7924612.0,758980.0,15.885484,13.539732,2.437215,10.441134,5
2,5ww2BF9slyYgNOk37BlC4u,Manuel Turizo,La Bachata,latin,La Bachata - Manuel Turizo,162637,124.980,4.828154,2.710617,reggaeton latin pop latin pop latino bachata male vocalist colombian pop colombia colombian,0.58300,0.835,0.679,0.000002,0.2180,-5.329,0.0364,0.850,4325531.0,365331.0,15.280046,12.808562,2.552568,11.840033,10
3,6Sq7ltF9Qa7SNFBsV5Cogx,Bad Bunny;Chencho Corleone,Me Porto Bonito,latin,"Me Porto Bonito - Bad Bunny, Chencho Corleone",178567,92.005,4.521843,2.976117,bad bunny reggaeton fire latin chencho corleone,0.09010,0.911,0.712,0.000027,0.0933,-5.105,0.0817,0.425,8928528.0,632558.0,16.004762,13.357529,2.715685,14.114955,5
4,5Eax0qFko2dh7Rl2lYs3bx,Bad Bunny,Efecto,latin,Efecto - Bad Bunny,213061,98.047,4.585447,3.551017,reggaeton latin,0.14100,0.801,0.475,0.000017,0.0639,-8.797,0.0516,0.234,6119925.0,420681.0,15.627061,12.949632,2.743910,14.547662,2



## 4.2 Helpers for lookups and normalization

Normalize titles/artists and resolve user history into dataset indices so downstream search hits the correct feature rows. This keeps text handling aligned with the hygiene used in notebooks 1–3.


In [152]:

import re
import unicodedata

def strip_accents(text: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", text) if not unicodedata.combining(ch))

REMOVE_BRACKETS_RE = re.compile(r"\([^)]*\)|\[[^\]]*\]")
NON_ALNUM_RE = re.compile(r"[^a-z0-9]+")
SPACE_RE = re.compile(r"\s+")

def normalize_text(value: str) -> str:
    if not isinstance(value, str):
        return ""
    text = value.lower().strip()
    text = REMOVE_BRACKETS_RE.sub(" ", text)
    text = strip_accents(text)
    text = NON_ALNUM_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip()

def _match_rows(df: pd.DataFrame, title: str, artist: str | None):
    title_mask = df["track_name"].str.lower().fillna("") == title.lower()
    artist_mask = (
        df["artists"].str.lower().fillna("").str.contains(artist.lower()) if artist else True
    )
    return title_mask & artist_mask

def find_track_index(title: str | None = None, artist: str | None = None, track_id: str | None = None):
    if track_id:
        idx = id_to_index.get(str(track_id))
        if idx is not None:
            return idx
    if not title:
        return None
    mask = _match_rows(full_df, title, artist)
    if not mask.any():
        normalized_title = normalize_text(title)
        normalized_artist = normalize_text(artist) if artist else None
        alt_df = full_df.assign(
            track_name=full_df["track_name"].map(normalize_text),
            artists=full_df["artists"].map(normalize_text),
        )
        mask = _match_rows(alt_df, normalized_title, normalized_artist)
    if mask.any():
        return mask[mask].index[0]
    return None

def resolve_history_inputs(history):
    '''
    Accepts a list of entries. Each entry can be:
    - dict with track_id or track_name (+ optional artist)
    - (title, artist) tuple/list
    - track_id string
    - title string (artist optional)
    Returns: (indices list, missing entries list)
    '''
    resolved: list[int] = []
    missing = []
    for entry in history:
        title = artist = track_id = None
        if isinstance(entry, dict):
            title = entry.get("track_name") or entry.get("title")
            artist = entry.get("artist") or entry.get("artists")
            track_id = entry.get("track_id")
        elif isinstance(entry, (list, tuple)):
            if len(entry) == 2:
                title, artist = entry
            elif len(entry) == 1:
                title = entry[0]
        elif isinstance(entry, str):
            track_id = entry if entry in id_to_index else None
            if track_id is None:
                title = entry

        idx = find_track_index(title=title, artist=artist, track_id=track_id)
        if idx is None:
            missing.append(entry)
        else:
            resolved.append(int(idx))
    return resolved, missing

def describe_tracks(indices):
    cols = ["track_id", "track_name", "artists", "track_genre"]
    return full_df.loc[indices, cols]



## 4.3 LSH index (from notebook 2)

Load or rebuild the random hyperplane LSH index for the fast search path. Hash buckets provide a candidate pool that we later re-rank with cosine to approximate the exact search.


In [153]:
class RandomHyperplaneLSH:
    def __init__(self, n_bits_per_table=16, n_tables=8, random_state=42):
        self.n_bits_per_table = n_bits_per_table
        self.n_tables = n_tables
        self.random_state = random_state
        self.hyperplanes = None
        self.lsh_bits = None
        self.tables = None
        self.bucket_maps = None

    def fit(self, X, precomputed_bits=None):
        X = np.asarray(X, dtype=np.float32)
        n_items, n_features = X.shape
        if precomputed_bits is not None:
            bits = np.asarray(precomputed_bits)
            if bits.ndim != 2 or bits.shape[0] != n_items:
                raise ValueError("precomputed_bits shape mismatch")
            self.n_bits_per_table = int(bits.shape[1] // self.n_tables)
            self.hyperplanes = None
            self.lsh_bits = bits.astype(np.uint8, copy=False)
        else:
            rng = np.random.RandomState(self.random_state)
            self.hyperplanes = rng.normal(
                size=(self.n_tables, self.n_bits_per_table, n_features)
            ).astype(np.float32)
            projections = np.einsum("tbl,nl->ntb", self.hyperplanes, X)
            self.lsh_bits = (projections >= 0).astype(np.uint8).reshape(n_items, -1)
        self._build_buckets()
        return self

    def _build_buckets(self):
        n_items = self.lsh_bits.shape[0]
        self.tables = self.lsh_bits.reshape(n_items, self.n_tables, self.n_bits_per_table)
        self.bucket_maps = [defaultdict(list) for _ in range(self.n_tables)]
        for idx in range(n_items):
            for t in range(self.n_tables):
                bucket = tuple(int(b) for b in self.tables[idx, t])
                self.bucket_maps[t][bucket].append(idx)

    def query_index(self, idx: int, max_candidates: int = 300):
        if self.tables is None:
            raise ValueError("Index not built")
        candidates = set()
        for t in range(self.n_tables):
            bucket = tuple(int(b) for b in self.tables[idx, t])
            candidates.update(self.bucket_maps[t].get(bucket, []))
        candidates.discard(idx)
        if not candidates:
            return np.array([], dtype=int)
        ordered = list(candidates)
        if len(ordered) > max_candidates:
            ordered = ordered[:max_candidates]
        return np.array(ordered, dtype=int)

    def query_vector(self, vec, max_candidates: int = 300):
        if self.hyperplanes is None:
            raise ValueError("Hyperplanes not stored (loaded from precomputed signatures)")
        vec = np.asarray(vec, dtype=np.float32)
        bits = (np.einsum("tbl,l->tb", self.hyperplanes, vec) >= 0).astype(np.uint8)
        candidates = set()
        for t in range(self.n_tables):
            bucket = tuple(int(b) for b in bits[t])
            candidates.update(self.bucket_maps[t].get(bucket, []))
        if not candidates:
            return np.array([], dtype=int)
        ordered = list(candidates)
        if len(ordered) > max_candidates:
            ordered = ordered[:max_candidates]
        return np.array(ordered, dtype=int)

    def __repr__(self):
        return (
            f"RandomHyperplaneLSH(tables={self.n_tables}, bits_per_table={self.n_bits_per_table}, "
            f"hyperplanes={'yes' if self.hyperplanes is not None else 'precomputed'})"
        )

# Load precomputed signatures if available, otherwise rebuild
if paths["lsh_signatures"].exists():
    lsh_npz = np.load(paths["lsh_signatures"], allow_pickle=True)
    pre_bits = lsh_npz["lsh_bits"]
    n_tables = int(lsh_npz["n_tables"])
    n_bits_per_table = int(lsh_npz["n_bits_per_table"])
    seed = int(lsh_npz["seed"]) if "seed" in lsh_npz.files else RANDOM_STATE
    lsh_index = RandomHyperplaneLSH(
        n_bits_per_table=n_bits_per_table,
        n_tables=n_tables,
        random_state=seed,
    ).fit(all_weighted, precomputed_bits=pre_bits)
else:
    lsh_index = RandomHyperplaneLSH(
        n_bits_per_table=16,
        n_tables=8,
        random_state=RANDOM_STATE,
    ).fit(all_weighted)

lsh_index


RandomHyperplaneLSH(tables=8, bits_per_table=16, hyperplanes=precomputed)


## 4.4 Cluster labels for reranking (from notebook 3)

Fit (or load) KMeans cluster labels on the weighted features. These act as a light prior so recommendations stay near the listener’s dominant clusters, echoing the clustering work in notebook 3.


In [154]:
CACHE_KMEANS = False
kmeans_cache_path = ARTIFACT_DIR / "kmeans_labels.npy"

if kmeans_cache_path.exists():
    cluster_labels = np.load(kmeans_cache_path)
else:
    n_clusters = full_df["track_genre"].nunique()
    kmeans = KMeans(
        n_clusters=n_clusters,
        n_init=10,
        random_state=RANDOM_STATE,
    )
    cluster_labels = kmeans.fit_predict(all_weighted)
    if CACHE_KMEANS:
        np.save(kmeans_cache_path, cluster_labels)

full_df["cluster_kmeans_main"] = cluster_labels
cluster_sizes = pd.Series(cluster_labels).value_counts().sort_values(ascending=False)
print(f"Clusters: {cluster_labels.max() + 1}; median size={cluster_sizes.median():.0f}")
cluster_sizes.head()

# Popularity prior (log-scaled plays) for light re-ranking
pop_series = full_df["log_playcount"].fillna(full_df["log_playcount"].median())
pop_mean, pop_std = pop_series.mean(), pop_series.std()


Clusters: 113; median size=434



## 4.5 Recommendation pipelines: exact vs. LSH

Core scoring for both modes. Exact uses full-catalog cosine; LSH uses hashed candidates plus cosine. Scores blend similarity, seed alignment, cluster prior, genre prior, popularity, and artist overlap, and we can toggle diversification depending on quality vs. variety needs.


In [155]:

from typing import Iterable

def build_user_profile(seed_indices, recency_bias: float = 1.1):
    weights = recency_bias ** np.arange(len(seed_indices))[::-1]
    profile_vec = np.average(all_weighted_unit[seed_indices], axis=0, weights=weights)
    profile_vec = profile_vec / np.clip(np.linalg.norm(profile_vec), 1e-12, None)
    cluster_prior = (
        pd.Series(cluster_labels[seed_indices])
        .value_counts(normalize=True)
        .to_dict()
    )
    seed_artists = (
        full_df.loc[seed_indices, "artists"]
        .str.lower()
        .str.split(";")
        .explode()
        .str.strip()
        .dropna()
        .unique()
    )
    seed_artists = set(seed_artists)
    genre_prior = (
        full_df.loc[seed_indices, "track_genre"]
        .value_counts(normalize=True)
        .to_dict()
    )
    return profile_vec, cluster_prior, seed_artists, genre_prior

def gather_candidates_lsh(seed_indices: Iterable[int], max_candidates_per_seed: int = 1200):
    counts = Counter()
    for idx in seed_indices:
        cands = lsh_index.query_index(idx, max_candidates=max_candidates_per_seed)
        for c in cands:
            counts[int(c)] += 1
    return counts

def score_candidates(seed_indices, candidate_indices, candidate_counts, profile_vec, cluster_prior, seed_artists, genre_prior):
    cand_vectors = all_weighted_unit[candidate_indices]

    sim_profile = cand_vectors @ profile_vec
    seed_unit = all_weighted_unit[seed_indices]
    sim_best_seed = cosine_similarity(cand_vectors, seed_unit).max(axis=1)

    cluster_ids = cluster_labels[candidate_indices]
    cluster_bonus = np.array([cluster_prior.get(int(c), 0.0) for c in cluster_ids])

    pop_vals = pop_series.iloc[candidate_indices].to_numpy()
    pop_z = np.clip((pop_vals - pop_mean) / pop_std, -1, 1)

    artists_lower = full_df.loc[candidate_indices, "artists"].str.lower()
    artist_match = artists_lower.apply(lambda a: any(sa in a for sa in seed_artists)).astype(float).to_numpy()

    genres = full_df.loc[candidate_indices, "track_genre"].astype(str)
    genre_bonus = np.array([genre_prior.get(g, 0.0) for g in genres])

    # Weights: lean heavily on cosine and seed match; moderate cluster + genre; small pop/artist nudges
    w_sim = 0.75
    w_seed = 0.05
    w_cluster = 0.05
    w_genre = 0.05
    w_pop = 0.05
    w_artist = 0.05

    score_sim = w_sim * sim_profile
    score_seed = w_seed * sim_best_seed
    score_cluster = w_cluster * cluster_bonus
    score_genre = w_genre * genre_bonus
    score_pop = w_pop * pop_z
    score_artist = w_artist * artist_match

    score = score_sim + score_seed + score_cluster + score_genre + score_pop + score_artist

    rec_df = pd.DataFrame(
        {
            "index": candidate_indices,
            "track_id": full_df.loc[candidate_indices, "track_id"].values,
            "track_name": full_df.loc[candidate_indices, "track_name"].values,
            "artists": full_df.loc[candidate_indices, "artists"].values,
            "track_genre": genres.values,
            "cluster": cluster_ids,
            "sim_profile": sim_profile,
            "sim_best_seed": sim_best_seed,
            "cluster_bonus": cluster_bonus,
            "genre_bonus": genre_bonus,
            "popularity_z": pop_z,
            "artist_match": artist_match,
            "score_sim": score_sim,
            "score_seed": score_seed,
            "score_cluster": score_cluster,
            "score_genre": score_genre,
            "score_pop": score_pop,
            "score_artist": score_artist,
            "score": score,
        }
    ).sort_values("score", ascending=False)
    rec_df["why"] = rec_df.apply(
        lambda row: (
            f"sim={row.score_sim:.2f}, seed={row.score_seed:.2f}, "
            f"cluster={row.score_cluster:.2f}, genre={row.score_genre:.2f}, "
            f"pop={row.score_pop:.2f}, artist={row.score_artist:.2f}"
        ),
        axis=1,
    )
    return rec_df.reset_index(drop=True)

def diversify_recs(rec_df: pd.DataFrame, top_n: int = 20, artist_cap: int | None = 2, cluster_cap: int | None = 2, diversify: bool = False):
    if not diversify:
        return rec_df.head(top_n)
    picks = []
    artist_counts = Counter()
    cluster_counts = Counter()
    for _, row in rec_df.iterrows():
        if artist_cap is not None and artist_counts[row["artists"]] >= artist_cap:
            continue
        if cluster_cap is not None and cluster_counts[row["cluster"]] >= cluster_cap:
            continue
        picks.append(row)
        artist_counts[row["artists"]] += 1
        cluster_counts[row["cluster"]] += 1
        if len(picks) >= top_n:
            break
    if not picks:
        return rec_df.head(top_n)
    return pd.DataFrame(picks).reset_index(drop=True)

def recommend_from_indices(
    seed_indices,
    mode: str = "exact",  # "exact" or "lsh"
    top_n: int = 20,
    max_candidates_per_seed: int = 1200,
    recency_bias: float = 1.1,
    artist_cap: int | None = 2,
    cluster_cap: int | None = 2,
    diversify: bool = False,
):
    if len(seed_indices) < 3:
        raise ValueError("Provide at least 3 seed tracks for a stable profile (ideally 10).")
    profile_vec, cluster_prior, seed_artists, genre_prior = build_user_profile(seed_indices, recency_bias=recency_bias)

    if mode == "exact":
        candidate_indices = np.setdiff1d(np.arange(len(all_weighted_unit)), seed_indices)
        candidate_counts = None
    elif mode == "lsh":
        candidate_counts = gather_candidates_lsh(seed_indices, max_candidates_per_seed=max_candidates_per_seed)
        candidate_indices = np.array(list(candidate_counts.keys()), dtype=int)
        if len(candidate_indices) == 0:
            candidate_indices = np.setdiff1d(np.arange(len(all_weighted_unit)), seed_indices)
            candidate_counts = None
    else:
        raise ValueError("mode must be 'exact' or 'lsh'")

    rec_df = score_candidates(seed_indices, candidate_indices, candidate_counts, profile_vec, cluster_prior, seed_artists, genre_prior)
    rec_df = diversify_recs(rec_df, top_n=top_n, artist_cap=artist_cap, cluster_cap=cluster_cap, diversify=diversify)
    return rec_df

def recommend(
    history,
    mode: str = "exact",
    top_n: int = 20,
    max_candidates_per_seed: int = 1200,
    recency_bias: float = 1.1,
    artist_cap: int | None = 2,
    cluster_cap: int | None = 2,
    diversify: bool = False,
):
    seed_indices, missing = resolve_history_inputs(history)
    rec_df = recommend_from_indices(
        seed_indices,
        mode=mode,
        top_n=top_n,
        max_candidates_per_seed=max_candidates_per_seed,
        recency_bias=recency_bias,
        artist_cap=artist_cap,
        cluster_cap=cluster_cap,
        diversify=diversify,
    )
    return {
        "seed_indices": seed_indices,
        "missing": missing,
        "recs": rec_df,
    }

def summarize_mix(seed_indices, rec_df):
    seed_clusters = pd.Series(cluster_labels[seed_indices]).value_counts(normalize=True)
    rec_clusters = rec_df["cluster"].value_counts(normalize=True)
    summary = pd.concat([seed_clusters, rec_clusters], axis=1)
    summary.columns = ["user_cluster_share", "rec_cluster_share"]
    return summary.fillna(0).sort_values("user_cluster_share", ascending=False)



## 4.6 Demo: exact cosine vs. LSH

Run a 5-10-song, genre-diverse history through both pipelines and inspect per-track score breakdowns (sim/seed/cluster/genre/pop/artist) and cluster mixes. Top-20 lists are shown without diversification to emphasize accuracy and highlight differences between exact and LSH.


In [169]:

demo_history = [
    {"track_name": "HUMBLE.", "artist": "Kendrick Lamar"},
    {"track_name": "All The Stars (with SZA)", "artist": "Kendrick Lamar"},
    {"track_name": "SICKO MODE", "artist": "Travis Scott"},
    {"track_name": "positions", "artist": "Ariana Grande"},
    {"track_name": "Shape of You", "artist": "Ed Sheeran"},
    {"track_name": "Die For You", "artist": "The Weeknd"},
    {"track_name": "ROCKSTAR", "artist": "DaBaby"},
    {"track_name": "BUTTERFLY EFFECT", "artist": "Travis Scott"},
    ]


exact_res = recommend(demo_history, mode="exact", top_n=20, diversify=False)
lsh_res = recommend(demo_history, mode="lsh", top_n=20, max_candidates_per_seed=1200, diversify=False)

print(f"Resolved {len(exact_res['seed_indices'])} seeds; missing={exact_res['missing']}")
print("Exact cosine (top 20):")
display(exact_res["recs"][
    [
        "track_name", "artists", "track_genre", "score", "score_sim", "score_seed", "score_cluster",
        "score_genre", "score_pop", "score_artist", "why",
    ]
].head(20))

print("LSH + cosine (top 20):")
display(lsh_res["recs"][
    [
        "track_name", "artists", "track_genre", "score", "score_sim", "score_seed", "score_cluster",
        "score_genre", "score_pop", "score_artist", "why",
    ]
].head(20))

print("Cluster mix (exact):")
display(summarize_mix(exact_res["seed_indices"], exact_res["recs"]))
print("Cluster mix (LSH):")
display(summarize_mix(exact_res["seed_indices"], lsh_res["recs"]))


Resolved 8 seeds; missing=[]
Exact cosine (top 20):


,track_name,artists,track_genre,score,score_sim,score_seed,score_cluster,score_genre,score_pop,score_artist,why
0,N95,Kendrick Lamar,hip-hop,0.772883,0.584666,0.044467,0.01875,0.02500,0.050000,0.05,"sim=0.58, seed=0.04, cluster=0.02, genre=0.03, pop=0.05, artist=0.05"
1,INDUSTRY BABY (feat. Jack Harlow),Lil Nas X;Jack Harlow,hip-hop,0.760160,0.618365,0.048046,0.01875,0.02500,0.050000,0.00,"sim=0.62, seed=0.05, cluster=0.02, genre=0.03, pop=0.05, artist=0.00"
2,Best Friend,Young Thug,hip-hop,0.757875,0.617875,0.046250,0.01875,0.02500,0.050000,0.00,"sim=0.62, seed=0.05, cluster=0.02, genre=0.03, pop=0.05, artist=0.00"
3,R.I.P.,Playboi Carti,hip-hop,0.757494,0.615054,0.048690,0.01875,0.02500,0.050000,0.00,"sim=0.62, seed=0.05, cluster=0.02, genre=0.03, pop=0.05, artist=0.00"
4,7 rings,Ariana Grande,dance,0.754016,0.594840,0.034176,0.01875,0.00625,0.050000,0.05,"sim=0.59, seed=0.03, cluster=0.02, genre=0.01, pop=0.05, artist=0.05"
5,Trap Queen,Fetty Wap,j-dance,0.725153,0.609689,0.046714,0.01875,0.00000,0.050000,0.00,"sim=0.61, seed=0.05, cluster=0.02, genre=0.00, pop=0.05, artist=0.00"
6,"break up with your girlfriend, i'm bored",Ariana Grande,dance,0.719625,0.559398,0.035227,0.01875,0.00625,0.050000,0.05,"sim=0.56, seed=0.04, cluster=0.02, genre=0.01, pop=0.05, artist=0.05"
7,The Search,NF,hip-hop,0.705449,0.568463,0.043235,0.01875,0.02500,0.050000,0.00,"sim=0.57, seed=0.04, cluster=0.02, genre=0.03, pop=0.05, artist=0.00"
8,WORKIN ME,Quavo,hip-hop,0.679955,0.542017,0.044188,0.01875,0.02500,0.050000,0.00,"sim=0.54, seed=0.04, cluster=0.02, genre=0.03, pop=0.05, artist=0.00"
9,WALK IN THE PARK,Jack Harlow,hip-hop,0.673094,0.553382,0.038463,0.00625,0.02500,0.050000,0.00,"sim=0.55, seed=0.04, cluster=0.01, genre=0.03, pop=0.05, artist=0.00"


LSH + cosine (top 20):


,track_name,artists,track_genre,score,score_sim,score_seed,score_cluster,score_genre,score_pop,score_artist,why
0,Nail Tech,Jack Harlow,hip-hop,0.637042,0.518325,0.037467,0.00625,0.025,0.050000,0.0,"sim=0.52, seed=0.04, cluster=0.01, genre=0.03, pop=0.05, artist=0.00"
1,SUVs (Black on Black),Jack Harlow;Pooh Shiesty,hip-hop,0.634458,0.516726,0.036482,0.00625,0.025,0.050000,0.0,"sim=0.52, seed=0.04, cluster=0.01, genre=0.03, pop=0.05, artist=0.00"
2,RAIN,Jack Harlow,hip-hop,0.621953,0.518942,0.035881,0.00625,0.025,0.035880,0.0,"sim=0.52, seed=0.04, cluster=0.01, genre=0.03, pop=0.04, artist=0.00"
3,Empty Lightning,Woesum;Oklou,club,0.594081,0.491931,0.039324,0.01875,0.000,0.044076,0.0,"sim=0.49, seed=0.04, cluster=0.02, genre=0.00, pop=0.04, artist=0.00"
4,NIKEYS PT. 2 IN MINT,Yxngxr1,sad,0.552573,0.471597,0.042163,0.00000,0.000,0.038813,0.0,"sim=0.47, seed=0.04, cluster=0.00, genre=0.00, pop=0.04, artist=0.00"
5,Gang Outside,kizaru;Milian Beatz;Lil Gotit,emo,0.458627,0.437462,0.042064,0.00000,0.000,-0.020899,0.0,"sim=0.44, seed=0.04, cluster=0.00, genre=0.00, pop=-0.02, artist=0.00"
6,Sitting In Fire,MASN,chill,0.456619,0.383423,0.030168,0.00000,0.000,0.043028,0.0,"sim=0.38, seed=0.03, cluster=0.00, genre=0.00, pop=0.04, artist=0.00"
7,absolute in doubt,Lil Peep;Wicca Phase Springs Eternal,emo,0.437499,0.353381,0.034118,0.00000,0.000,0.050000,0.0,"sim=0.35, seed=0.03, cluster=0.00, genre=0.00, pop=0.05, artist=0.00"
8,As Mais Braba,Tasha & Tracie;Ashira,funk,0.430757,0.332751,0.029255,0.01875,0.000,0.050000,0.0,"sim=0.33, seed=0.03, cluster=0.02, genre=0.00, pop=0.05, artist=0.00"
9,Dangerous State of Mind,Chri$tian Gate$,sad,0.403243,0.319397,0.033845,0.00000,0.000,0.050000,0.0,"sim=0.32, seed=0.03, cluster=0.00, genre=0.00, pop=0.05, artist=0.00"


Cluster mix (exact):


,user_cluster_share,rec_cluster_share
94,0.375,0.7
4,0.250,0.0
1,0.125,0.3
73,0.125,0.0
96,0.125,0.0


Cluster mix (LSH):


,user_cluster_share,rec_cluster_share
94,0.375,0.25
4,0.250,0.05
1,0.125,0.35
73,0.125,0.00
96,0.125,0.00
70,0.000,0.30
48,0.000,0.05



## 4.7 Conclusion

We converged on weights that put 75% of the score on cosine (0.75 sim + 0.05 seed) and keep the remaining 25% as small priors: cluster 0.05, genre 0.05, popularity 0.05, artist overlap 0.05. This preserves the strong neighbors seen in notebook 1 while adding mild guidance from user-preferred clusters/genres and mainstream signals, without overpowering similarity. Exact mode searches the full catalog; LSH trades quality for speed with larger candidate pools. The demo and cluster mixes show how these choices keep obvious neighbors (e.g., artist and style matches) high while still allowing some diversity; you can dial diversification back on if you need more variety at the cost of tight relevance.
